# N2 · IsaacLab 真跑指引 (需 NV GPU)

> 配套 11.6-L2 · 这是一个**指引** notebook (不在 CPU 上跑真 IsaacLab, 需 NVIDIA GPU + Isaac Sim)。
> 复用本人 AntBot 实战 + WSL2 踩坑经验 (详见 `src/isaaclab_notes.md`)。
> 下面的 code cell 只做**环境检测 + 打印命令**, 不实际启动 IsaacLab (保证 nbconvert 安全)。

## 1. 结论先行 (本人踩坑)
> ❌ **WSL2 + NV591 + Ubuntu22.04 + IsaacLab 5.1.0 → 失败** (Vulkan 不支持, headless 也死锁)
> ✅ **Windows 原生 + conda `env_isaaclab` + Isaac Sim 5.1 → 成功跑 AntBot**
> 教训: IsaacLab 对图形栈 (Vulkan/驱动) 极挑剔; **用 Windows 原生 / 原生 Ubuntu + 官方驱动**。

## 2. 环境检测 (只读, 不启动 IsaacLab)

In [1]:
import shutil, importlib.util, platform
print('平台:', platform.system(), platform.release())
# 检测 NVIDIA GPU
has_smi = shutil.which('nvidia-smi') is not None
print('nvidia-smi 可用:', has_smi, '(IsaacLab 需 NVIDIA GPU)')
# 检测 isaaclab / isaacsim (一般不在本环境)
for mod in ['isaaclab', 'isaacsim', 'omni']:
    print(f'  {mod} 模块:', '在' if importlib.util.find_spec(mod) else '不在 (正常, 真跑需专门安装)')
print('\n→ 本课 CPU 环境不装 IsaacLab; 下面给真跑命令, 在你的 GPU 机器上执行。')

平台: Windows 11
nvidia-smi 可用: True (IsaacLab 需 NVIDIA GPU)
  isaaclab 模块: 不在 (正常, 真跑需专门安装)
  isaacsim 模块: 不在 (正常, 真跑需专门安装)
  omni 模块: 不在 (正常, 真跑需专门安装)

→ 本课 CPU 环境不装 IsaacLab; 下面给真跑命令, 在你的 GPU 机器上执行。


## 3. 真跑流程 (在 NV GPU 机器上, Windows 原生)

In [2]:
guide = r'''
========== IsaacLab 真跑流程 (Windows 原生, 复用 AntBot 经验) ==========

[步骤 1] 验证最小启动 (先确认环境, 别一上来就训练):
  isaaclab.bat -p scripts\tutorials\00_sim\create_empty.py
  关键确认: [INFO][AppLauncher]: Using device: cuda:0   <- GPU 在用

[步骤 2] 训练 AntBot (并行 RL):
  isaaclab.bat -p scripts\reinforcement_learning\rsl_rl\train.py --task=Isaac-Ant-v0 --headless
  --headless = 无窗口纯算(快, 训练用); 去掉则开窗可视化(慢, debug)

[步骤 3] 回放/评估训出的策略:
  isaaclab.bat -p scripts\reinforcement_learning\rsl_rl\play.py --task=Isaac-Ant-v0

[踩坑提醒] (详见 src/isaaclab_notes.md):
  - 避开 WSL2 (Vulkan 支持差, headless 也死锁); 用 Windows 原生 / 原生 Ubuntu+官方驱动
  - 驱动版本要匹配 Isaac Sim; 卡住看 .../isaacsim/kit/logs/.../kit_*.log (日志停增=死锁)
  - 首次启动会联网同步扩展注册表; 国内配代理/镜像
'''
print(guide)


========== IsaacLab 真跑流程 (Windows 原生, 复用 AntBot 经验) ==========

[步骤 1] 验证最小启动 (先确认环境, 别一上来就训练):
  isaaclab.bat -p scripts\tutorials\00_sim\create_empty.py
  关键确认: [INFO][AppLauncher]: Using device: cuda:0   <- GPU 在用

[步骤 2] 训练 AntBot (并行 RL):
  isaaclab.bat -p scripts\reinforcement_learning\rsl_rl\train.py --task=Isaac-Ant-v0 --headless
  --headless = 无窗口纯算(快, 训练用); 去掉则开窗可视化(慢, debug)

[步骤 3] 回放/评估训出的策略:
  isaaclab.bat -p scripts\reinforcement_learning\rsl_rl\play.py --task=Isaac-Ant-v0

[踩坑提醒] (详见 src/isaaclab_notes.md):
  - 避开 WSL2 (Vulkan 支持差, headless 也死锁); 用 Windows 原生 / 原生 Ubuntu+官方驱动
  - 驱动版本要匹配 Isaac Sim; 卡住看 .../isaacsim/kit/logs/.../kit_*.log (日志停增=死锁)
  - 首次启动会联网同步扩展注册表; 国内配代理/镜像



## 4. 把 N1 (DR) 和真 IsaacLab 连起来

- **N1 你做的**: GPU-free 玩具, 随机化「目标分布」演示 DR 覆盖→泛化。
- **真 IsaacLab 里**: 在 `--task` 的环境配置里随机化**摩擦/质量/光照/传感器噪声/电机强度**等 (Isaac 提供 DR 接口), 机制和 N1 完全一样 (覆盖真实的不确定性)。
- **sim 内 RL** (L4): rsl_rl 在并行环境里跑 PPO; 你 AntBot 跑的就是这个。

> 你已经实际跑过 AntBot (sim 内 RL) + 趟过配置坑。加上本模块的 DR/gap 认知, 你对 sim2real 是「会跑 + 懂原理」的完整掌握。

## 5. 反思 (11.6 收口)

- 真 IsaacLab 需 NV GPU; 本课用 GPU-free 玩具 (N1) 讲清 DR 机制, 真跑指引 (本 notebook) 复用你 AntBot 经验。
- **配置是第一道坎**: WSL2 Vulkan 坑 → Windows 原生 (你的一手教训, `isaaclab_notes.md`)。
- DR 机制 sim/真一致: 随机化覆盖真实不确定性 → 泛化。

> **M11.6 收口**: 仿真便宜安全并行; sim2real gap=分布偏移; DR 用覆盖弥合; sim 内 RL 喂饱样本 (AntBot)。
> **交棒 M11.7「embodied-graduation」**: Module 11 capstone — 端到端 mini-VLA 装配 + 评测 (LIBERO/CALVIN 思路) + 研究 gap。下一专题 `embodied-graduation`。